In [12]:
from pathlib import Path
from datetime import datetime, timezone

import sys
import json
import platform
import random
import time

import numpy as np
import pandas as pd

import torch
import torchvision
import sklearn
import optuna
import tensorboard
import tqdm

In [13]:
PROJECT_DIR = Path(r"C:\Users\Joseph (YO)\JGM\U\Semestre_10\InteligenciaArtificial\Proyecto_Yuca")

DATASET_DIR = (
    PROJECT_DIR / "dataset" / "raw" / "cassava_dataset"
)

CONFIGS_DIR = PROJECT_DIR / "configs"

REPORTS_DIR = PROJECT_DIR / "reports"

MODELS_DIR = PROJECT_DIR / "models"

EXPERIMENTS_DIR = PROJECT_DIR / "experiments"

LOGS_DIR = PROJECT_DIR / "logs"

if str(PROJECT_DIR) not in sys.path:

    sys.path.insert(0, str(PROJECT_DIR))

In [14]:
directories = [

    MODELS_DIR / "baseline",

    MODELS_DIR / "checkpoints",

    MODELS_DIR / "optimized",

    EXPERIMENTS_DIR / "baseline",

    EXPERIMENTS_DIR / "hyperband",

    EXPERIMENTS_DIR / "cross_validation",

    LOGS_DIR / "tensorboard"

]

for directory in directories:

    directory.mkdir(
        parents=True,
        exist_ok=True
    )

    print(
        "Directorio disponible:",
        directory.relative_to(PROJECT_DIR)
    )

Directorio disponible: models\baseline
Directorio disponible: models\checkpoints
Directorio disponible: models\optimized
Directorio disponible: experiments\baseline
Directorio disponible: experiments\hyperband
Directorio disponible: experiments\cross_validation
Directorio disponible: logs\tensorboard


In [15]:
versions = {

    "Python": platform.python_version(),

    "PyTorch": torch.__version__,

    "Torchvision": torchvision.__version__,

    "NumPy": np.__version__,

    "Pandas": pd.__version__,

    "Scikit-learn": sklearn.__version__,

    "Optuna": optuna.__version__,

    "TensorBoard": tensorboard.__version__,

    "tqdm": tqdm.__version__

}

df_versions = pd.DataFrame(

    versions.items(),

    columns=["Library", "Version"]

)

display(df_versions)

,Library,Version
0,Python,3.11.0
1,PyTorch,2.5.1+cu121
2,Torchvision,0.20.1+cu121
3,NumPy,2.4.6
4,Pandas,3.1.1
5,Scikit-learn,1.9.1
6,Optuna,5.1.0
7,TensorBoard,2.21.0
8,tqdm,4.70.1


In [16]:
CUDA_AVAILABLE = torch.cuda.is_available()

DEVICE = torch.device(

    "cuda" if CUDA_AVAILABLE else "cpu"

)

print("Dispositivo seleccionado:", DEVICE)

print("CUDA disponible:", CUDA_AVAILABLE)

print("Versión CUDA de PyTorch:", torch.version.cuda)

print("Procesador:", platform.processor())

if CUDA_AVAILABLE:

    print(
        "Número de GPU:",
        torch.cuda.device_count()
    )

    print(
        "GPU:",
        torch.cuda.get_device_name(0)
    )

    gpu_properties = torch.cuda.get_device_properties(0)

    gpu_memory_gb = (
        gpu_properties.total_memory / 1024**3
    )

    print(
        f"Memoria GPU: {gpu_memory_gb:.2f} GiB"
    )

Dispositivo seleccionado: cuda
CUDA disponible: True
Versión CUDA de PyTorch: 12.1
Procesador: Intel64 Family 6 Model 140 Stepping 1, GenuineIntel
Número de GPU: 1
GPU: NVIDIA GeForce RTX 3050 Ti Laptop GPU
Memoria GPU: 4.00 GiB


In [17]:
assert torch.cuda.is_available(), (
    "PyTorch todavía no detecta CUDA."
)

x = torch.randn(
    1024,
    1024,
    device="cuda"
)

y = x @ x

torch.cuda.synchronize()

print("Dispositivo:", y.device)
print("Dimensiones:", y.shape)

print(
    "Memoria utilizada:",
    round(torch.cuda.memory_allocated() / 1024**2, 2),
    "MiB"
)

del x, y

torch.cuda.empty_cache()

Dispositivo: cuda:0
Dimensiones: torch.Size([1024, 1024])
Memoria utilizada: 16.12 MiB


In [18]:
def load_json(filepath):

    with open(
        filepath,
        "r",
        encoding="utf-8"
    ) as f:

        return json.load(f)


dataset_config = load_json(

    CONFIGS_DIR / "dataset_config.json"

)

preprocessing_config = load_json(

    CONFIGS_DIR / "preprocessing_config.json"

)

dataloader_config = load_json(

    CONFIGS_DIR / "dataloader_config.json"

)

federated_config = load_json(

    CONFIGS_DIR / "federated_config.json"

)

In [19]:
SEED = dataset_config["seed"]

NUM_CLASSES = dataset_config[
    "dataset_statistics"
]["num_classes"]

N_SPLITS = dataset_config[
    "cross_validation"
]["n_splits"]

CLASS_MAPPING = dataset_config["class_mapping"]

IMAGE_SIZE = tuple(
    preprocessing_config["image_size"]
)

BATCH_SIZE = 4

EFFECTIVE_BATCH_SIZE = 32

GRADIENT_ACCUMULATION_STEPS = (
    EFFECTIVE_BATCH_SIZE // BATCH_SIZE
)

NUM_WORKERS = dataloader_config["num_workers"]

NUM_CLIENTS = federated_config["num_clients"]

print("Semilla:", SEED)

print("Número de clases:", NUM_CLASSES)

print("Número de folds:", N_SPLITS)

print("Tamaño de imagen:", IMAGE_SIZE)

print("Batch size:", BATCH_SIZE)

print("Clientes federados:", NUM_CLIENTS)

Semilla: 42
Número de clases: 5
Número de folds: 3
Tamaño de imagen: (224, 224)
Batch size: 4
Clientes federados: 2


In [20]:
from src.utils import set_seed, get_device

set_seed(SEED)

DEVICE = get_device()

print("Semillas configuradas.")

print("Dispositivo:", DEVICE)

Semillas configuradas.
Dispositivo: cuda


In [21]:
x = torch.rand(

    16,

    3,

    224,

    224,

    device=DEVICE

)

print("Dimensiones:", x.shape)

print("Dispositivo:", x.device)

print("Tipo:", x.dtype)

Dimensiones: torch.Size([16, 3, 224, 224])
Dispositivo: cuda:0
Tipo: torch.float32


In [22]:
a = torch.randn(

    2048,

    2048,

    device=DEVICE

)

b = torch.randn(

    2048,

    2048,

    device=DEVICE

)

# Calentamiento inicial
for _ in range(3):

    result = a @ b

if DEVICE.type == "cuda":

    torch.cuda.synchronize()

start = time.perf_counter()

result = a @ b

if DEVICE.type == "cuda":

    torch.cuda.synchronize()

elapsed = time.perf_counter() - start

print("Dispositivo:", result.device)

print("Resultado:", result.shape)

print(f"Tiempo: {elapsed:.4f} segundos")

del a, b, result, x

Dispositivo: cuda:0
Resultado: torch.Size([2048, 2048])
Tiempo: 0.0334 segundos


C:\Users\Joseph (YO)\AppData\Local\Temp\ipykernel_22280\3440764974.py:24: UserWarning: Deterministic behavior was enabled with either `torch.use_deterministic_algorithms(True)` or `at::Context::setDeterministicAlgorithms(true)`, but this operation is not deterministic because it uses CuBLAS and you have CUDA >= 10.2. To enable deterministic behavior in this case, you must set an environment variable before running your PyTorch application: CUBLAS_WORKSPACE_CONFIG=:4096:8 or CUBLAS_WORKSPACE_CONFIG=:16:8. For more information, go to https://docs.nvidia.com/cuda/cublas/index.html#results-reproducibility (Triggered internally at C:\actions-runner\_work\pytorch\pytorch\builder\windows\pytorch\aten\src\ATen\Context.cpp:208.)
  result = a @ b
C:\Users\Joseph (YO)\AppData\Local\Temp\ipykernel_22280\3440764974.py:32: UserWarning: Deterministic behavior was enabled with either `torch.use_deterministic_algorithms(True)` or `at::Context::setDeterministicAlgorithms(true)`, but this operation is no

In [23]:
from src.preprocessing import build_transforms

from src.data_generator import create_fold_loaders

In [24]:
df_splits = pd.read_csv(

    REPORTS_DIR / "dataset_splits.csv"

)

train_transform, eval_transform = build_transforms(

    CONFIGS_DIR / "preprocessing_config.json"

)

train_loader, val_loader = create_fold_loaders(

    df_splits=df_splits,

    dataset_dir=DATASET_DIR,

    class_mapping=CLASS_MAPPING,

    train_transform=train_transform,

    eval_transform=eval_transform,

    fold=1,

    batch_size=BATCH_SIZE,

    num_workers=NUM_WORKERS,

    seed=SEED

)

print(
    "Imágenes de entrenamiento:",
    len(train_loader.dataset)
)

print(
    "Imágenes de validación:",
    len(val_loader.dataset)
)

Imágenes de entrenamiento: 12124
Imágenes de validación: 6063


In [25]:
images, labels = next(iter(train_loader))

images = images.to(DEVICE)

labels = labels.to(DEVICE)

print("Dimensiones de imágenes:", images.shape)

print("Dimensiones de etiquetas:", labels.shape)

print("Dispositivo de imágenes:", images.device)

print("Dispositivo de etiquetas:", labels.device)

assert images.device.type == DEVICE.type

assert labels.device.type == DEVICE.type

assert torch.isfinite(images).all()

print("Data Generator compatible con el dispositivo.")

Dimensiones de imágenes: torch.Size([4, 3, 224, 224])
Dimensiones de etiquetas: torch.Size([4])
Dispositivo de imágenes: cuda:0
Dispositivo de etiquetas: cuda:0
Data Generator compatible con el dispositivo.


In [26]:
from torch.utils.tensorboard import SummaryWriter

writer = SummaryWriter(

    log_dir=str(
        LOGS_DIR / "tensorboard" / "environment_test"
    )

)

for epoch in range(1, 6):

    simulated_loss = 1 / epoch

    simulated_accuracy = 1 - (0.5 / epoch)

    writer.add_scalar(

        "Demo/Loss",

        simulated_loss,

        epoch

    )

    writer.add_scalar(

        "Demo/Accuracy",

        simulated_accuracy,

        epoch

    )

writer.close()

print("Registro de prueba generado.")

Registro de prueba generado.


In [27]:
def objective(trial):

    x = trial.suggest_float(
        "x",
        -10,
        10
    )

    for epoch in range(1, 10):

        simulated_loss = (
            (x - 2) ** 2 + 1 / epoch
        )

        trial.report(

            simulated_loss,

            step=epoch

        )

        if trial.should_prune():

            raise optuna.TrialPruned()

    return simulated_loss

In [28]:
pruner = optuna.pruners.HyperbandPruner(

    min_resource=1,

    max_resource=9,

    reduction_factor=3

)

sampler = optuna.samplers.RandomSampler(

    seed=SEED

)

study = optuna.create_study(

    study_name="environment_test",

    direction="minimize",

    sampler=sampler,

    pruner=pruner

)

study.optimize(

    objective,

    n_trials=20

)

print("Mejor configuración:", study.best_params)

print("Mejor valor:", study.best_value)

[I 2026-11-24 01:33:03,928] A new study created in memory with name: environment_test
[I 2026-11-24 01:33:03,930] Trial 0 finished with value: 20.443974314855684 and parameters: {'x': -2.50919762305275}. Best is trial 0 with value: 20.443974314855684.
[I 2026-11-24 01:33:03,931] Trial 1 pruned. 
[I 2026-11-24 01:33:03,933] Trial 2 finished with value: 7.080071381076146 and parameters: {'x': 4.639878836228101}. Best is trial 2 with value: 7.080071381076146.
[I 2026-11-24 01:33:03,935] Trial 3 finished with value: 0.11183097697095132 and parameters: {'x': 1.973169683940732}. Best is trial 3 with value: 0.11183097697095132.
[I 2026-11-24 01:33:03,936] Trial 4 finished with value: 78.95889016494411 and parameters: {'x': -6.87962719115127}. Best is trial 3 with value: 0.11183097697095132.
[I 2026-11-24 01:33:03,938] Trial 5 pruned. 
[I 2026-11-24 01:33:03,940] Trial 6 pruned. 
[I 2026-11-24 01:33:03,941] Trial 7 pruned. 
[I 2026-11-24 01:33:03,942] Trial 8 finished with value: 0.11160841158

Mejor configuración: {'x': 2.0223002348641756}
Mejor valor: 0.1116084115861085


In [29]:
df_trials = study.trials_dataframe()

display(df_trials)

display(
    df_trials["state"].value_counts()
)

,number,value,datetime_start,datetime_complete,duration,params_x,system_attrs_completed_rung_0,system_attrs_completed_rung_1,system_attrs_completed_rung_2,state
0,0,20.443974,2026-11-24 01:33:03.929977,2026-11-24 01:33:03.930979,0 days 00:00:00.001002,-2.509198,21.332863,20.666197,20.443974,COMPLETE
1,1,50.200210,2026-11-24 01:33:03.931975,2026-11-24 01:33:03.931975,0 days 00:00:00.000000,9.014286,50.200210,NaN,NaN,PRUNED
2,2,7.080071,2026-11-24 01:33:03.932977,2026-11-24 01:33:03.933978,0 days 00:00:00.001001,4.639879,7.080071,NaN,NaN,COMPLETE
3,3,0.111831,2026-11-24 01:33:03.934978,2026-11-24 01:33:03.935978,0 days 00:00:00.001000,1.973170,1.000720,0.334053,0.111831,COMPLETE
4,4,78.958890,2026-11-24 01:33:03.935978,2026-11-24 01:33:03.936975,0 days 00:00:00.000997,-6.879627,79.181112,78.958890,NaN,COMPLETE
5,5,79.856346,2026-11-24 01:33:03.937977,2026-11-24 01:33:03.938979,0 days 00:00:00.001002,-6.880110,79.856346,NaN,NaN,PRUNED
6,6,117.802682,2026-11-24 01:33:03.939977,2026-11-24 01:33:03.940977,0 days 00:00:00.001000,-8.838328,117.802682,NaN,NaN,PRUNED
7,7,29.339896,2026-11-24 01:33:03.941976,2026-11-24 01:33:03.941976,0 days 00:00:00.000000,7.323523,29.339896,NaN,NaN,PRUNED
8,8,0.111608,2026-11-24 01:33:03.942976,2026-11-24 01:33:03.942976,0 days 00:00:00.000000,2.022300,0.333831,0.111608,NaN,COMPLETE
9,9,5.005206,2026-11-24 01:33:03.943979,2026-11-24 01:33:03.943979,0 days 00:00:00.000000,4.161452,5.671873,5.005206,NaN,PRUNED


state
PRUNED      15
COMPLETE     5
Name: count, dtype: int64

In [30]:
training_config = {
    "seed": SEED,

    "num_classes": NUM_CLASSES,

    "image_size": list(IMAGE_SIZE),

    "architectures": [
        "mobilenet_v3_large",
        "efficientnet_b0",
        "resnet50",
        "vit_b_16",
        "swin_t"
    ],

    "transfer_learning": {
        "pretrained": True,
        "freeze_backbone": True,
        "partial_finetuning": True
    },

    "training": {
        "batch_size": BATCH_SIZE,
        "effective_batch_size": 32,
        "gradient_accumulation_steps": 8,
        "num_workers": NUM_WORKERS,
        "baseline_epochs": 3,
        "max_finetuning_epochs": 10,
        "optimizer": "AdamW",
        "learning_rate": 0.001,
        "weight_decay": 0.0001,
        "dropout": 0.2,
        "loss_function": "CrossEntropyLoss",
        "class_weighting": True,
        "class_weight_method": "balanced",
        "mixed_precision": DEVICE.type == "cuda"
    },

    "model_settings": {
        "mobilenet_v3_large": {
            "batch_size": 16,
            "gradient_accumulation_steps": 2
        },
        "efficientnet_b0": {
            "batch_size": 16,
            "gradient_accumulation_steps": 2
        },
        "resnet50": {
            "batch_size": 8,
            "gradient_accumulation_steps": 4
        },
        "vit_b_16": {
            "batch_size": 4,
            "gradient_accumulation_steps": 8
        },
        "swin_t": {
            "batch_size": 4,
            "gradient_accumulation_steps": 8
        }
    },

    "evaluation": {
        "primary_metric": "f1_macro",
        "secondary_metrics": [
            "accuracy",
            "precision_macro",
            "recall_macro"
        ],
        "n_splits": N_SPLITS
    },

    "optimization": {
        "framework": "Optuna",
        "algorithm": "Hyperband",
        "min_resource": 1,
        "max_resource": 9,
        "reduction_factor": 3,
        "n_trials": 12,
        "validation_fold": 1
    }
}

with open(
    CONFIGS_DIR / "training_config.json",
    "w",
    encoding="utf-8"
) as f:

    json.dump(
        training_config,
        f,
        indent=4
    )

print("Configuración de entrenamiento guardada.")

Configuración de entrenamiento guardada.


In [31]:
training_environment = {

    "created_at": datetime.now(
        timezone.utc
    ).isoformat(),

    "platform": platform.platform(),

    "libraries": versions,

    "device": str(DEVICE),

    "cuda_available": CUDA_AVAILABLE,

    "cuda_version": torch.version.cuda,

    "gpu_name": (

        torch.cuda.get_device_name(0)

        if CUDA_AVAILABLE

        else None

    ),

    "gpu_memory_gib": (

        torch.cuda.get_device_properties(0)
        .total_memory / 1024**3

        if CUDA_AVAILABLE

        else None

    )

}

with open(

    CONFIGS_DIR / "training_environment.json",

    "w",

    encoding="utf-8"

) as f:

    json.dump(

        training_environment,

        f,

        indent=4

    )

print("Entorno de entrenamiento registrado.")

Entorno de entrenamiento registrado.
